# 📄 FILE: 04_inference.py

## 🎯 Chức năng chính
Đây là bước cuối cùng và thú vị nhất: **Kiểm thử thực tế (Inference)**.
Script này sẽ đóng vai trò là giao diện "Trợ lý ảo", cho phép bạn nhập triệu chứng bệnh vào và nhận lại kết quả chẩn đoán từ mô hình AI đã huấn luyện.

## ⚙️ Quy trình xử lý
1.  **Load Architecture:** Tự động import cấu trúc mô hình (`DiabetesHybridModel`) từ file `02_model_architecture.py`.
2.  **Load Weights:** Tải bộ trọng số thông minh nhất (`best_model.pth`) mà bạn vừa train được ở bước trước.
3.  **Predict Loop:**
    * Nhận văn bản input từ bàn phím.
    * Xử lý qua Tokenizer (ViBERT) và tính điểm Từ điển.
    * Đưa vào mô hình để tính toán xác suất (%).
    * In kết quả và lời khuyên ra màn hình.

In [1]:
# ==============================================================================
# FILE: 04_inference.py
# CHỨC NĂNG: Chat với mô hình AI để chẩn đoán bệnh
# ==============================================================================

In [2]:
import torch
from transformers import AutoTokenizer
import importlib
import sys
import os

In [3]:
# --- BƯỚC 1: IMPORT DYNAMIC TỪ FILE 02 ---
try:
    arch = importlib.import_module("02_model_architecture")
    DiabetesDataset = arch.DiabetesDataset
    DiabetesHybridModel = arch.DiabetesHybridModel
    MODEL_NAME = arch.MODEL_NAME
    MAX_LEN = arch.MAX_LEN
    print("Đã import cấu trúc mô hình từ File 02.")
except ImportError:
    print("❌ Lỗi: Không tìm thấy file '02_model_architecture.py'.")
    print("👉 Hãy chắc chắn bạn đang chạy file này trong cùng thư mục dự án.")
    # exit() # Nếu chạy trong Notebook thì comment dòng này lại

Đã import cấu trúc mô hình từ File 02.


In [4]:
# --- CẤU HÌNH THIẾT BỊ ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Đang chạy trên thiết bị: {DEVICE}")

⚙️ Đang chạy trên thiết bị: cuda


In [ ]:
# --- BƯỚC 2: LOAD MODEL & TOKENIZER ---
print("⏳ Đang khởi tạo mô hình...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DiabetesHybridModel(n_classes=2)

model_path = '../models/nlp/NLP_model.pth'

if os.path.exists(model_path):
    # Load trọng số đã train vào mô hình
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval() # Chuyển sang chế độ dự đoán (tắt Dropout)
    print(f"✅ Đã nạp thành công trọng số từ '{model_path}'!")
else:
    print(f"❌ Lỗi: Không tìm thấy file '{model_path}'.")
    print("👉 Bạn cần chạy File 03 để train mô hình trước đã!")
    # exit()

⏳ Đang khởi tạo mô hình...


config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/581M [00:00<?, ?B/s]

✅ Đã nạp thành công trọng số từ '../models/nlp/NLP_model.pth'!


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Exception ignored in: <function tqdm.__del__ at 0x0000023B521A3A30>
Traceback (most recent call last):
  File "d:\IT\HK1_Y4\Class\Project_1\Project_2\.venv\lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\IT\HK1_Y4\Class\Project_1\Project_2\.venv\lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm' object has no attribute 'disp'


In [7]:
# --- BƯỚC 3: HÀM DỰ ĐOÁN ---
# Tạo dataset giả để dùng lại hàm tính điểm từ điển (extract_weighted_features)
# Lưu ý: Phải có file 'diabetes_keywords.json' trong thư mục
try:
    helper_ds = DiabetesDataset([], [], [], 'diabetes_keywords.json', tokenizer)
except:
    print("⚠️ Cảnh báo: Không tìm thấy 'diabetes_keywords.json'. Điểm từ điển sẽ bằng 0.")
    helper_ds = DiabetesDataset([], [], [], '', tokenizer)

def predict_symptom(text):
    """Hàm nhận text -> Trả về nhãn dự đoán và độ tin cậy"""
    
    # 1. Xử lý văn bản cho ViBERT
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    
    # 2. Tính điểm từ điển
    dict_feat = helper_ds.extract_weighted_features(text)
    
    # 3. Đưa dữ liệu lên GPU/CPU
    input_ids = encoding['input_ids'].flatten().unsqueeze(0).to(DEVICE)
    attention_mask = encoding['attention_mask'].flatten().unsqueeze(0).to(DEVICE)
    dict_features = torch.tensor([dict_feat], dtype=torch.float).to(DEVICE)

    # 4. Mô hình suy luận
    with torch.no_grad(): # Không tính gradient để tiết kiệm nhớ
        outputs = model(input_ids, attention_mask, dict_features)
        
        # Tính xác suất (Softmax)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        confidence, prediction = torch.max(probs, dim=1)

    return prediction.item(), confidence.item()

In [8]:
# --- BƯỚC 4: GIAO DIỆN CHAT ---
print("\n" + "="*50)
print("🤖 TRỢ LÝ AI: SẴN SÀNG CHẨN ĐOÁN")
print("👉 Gõ 'exit' để thoát chương trình.")
print("="*50 + "\n")

while True:
    try:
        user_input = input("💬 Nhập triệu chứng: ")
        
        if user_input.lower() in ['exit', 'quit', 'thoat']:
            print("👋 Tạm biệt!")
            break
        
        if not user_input.strip():
            continue

        # Gọi hàm dự đoán
        label, conf = predict_symptom(user_input)
        
        # --- PHẦN HIỂN THỊ KẾT QUẢ (ĐÃ CẬP NHẬT) ---
        print("-" * 50)
        print(f"📝 INPUT: \"{user_input}\"")  # <--- Đã thêm dòng này
        print("-" * 50)
        
        if label == 1:
            print(f"🔴 KẾT QUẢ: CÓ NGUY CƠ CAO (Outcome=1)")
            print(f"📊 Độ tin cậy: {conf*100:.2f}%")
            print("💡 Lời khuyên: Các triệu chứng này khớp với dấu hiệu bệnh.")
            print("   Bạn nên đi khám bác sĩ chuyên khoa Nội tiết.")
        else:
            print(f"🟢 KẾT QUẢ: BÌNH THƯỜNG (Outcome=0)")
            print(f"📊 Độ tin cậy: {conf*100:.2f}%")
            print("💡 Lời khuyên: Chưa thấy dấu hiệu bệnh rõ ràng.")
            print("   Hãy duy trì lối sống lành mạnh nhé.")
        print("=" * 50 + "\n")
        
    except KeyboardInterrupt:
        print("\n👋 Đã dừng chương trình.")
        break
    except Exception as e:
        print(f"⚠️ Có lỗi xảy ra: {e}")
        break


🤖 TRỢ LÝ AI: SẴN SÀNG CHẨN ĐOÁN
👉 Gõ 'exit' để thoát chương trình.

--------------------------------------------------
📝 INPUT: ""Bác sĩ bảo đường huyết của tôi hơi cao nhưng chưa đến mức tiểu đường, chỉ cần ăn kiêng là được.""
--------------------------------------------------
🟢 KẾT QUẢ: BÌNH THƯỜNG (Outcome=0)
📊 Độ tin cậy: 99.87%
💡 Lời khuyên: Chưa thấy dấu hiệu bệnh rõ ràng.
   Hãy duy trì lối sống lành mạnh nhé.

--------------------------------------------------
📝 INPUT: ""Tôi đi tiểu đường... à nhầm, đi tiểu bình thường, đường sá dạo này đông quá làm tôi nói nhịu.""
--------------------------------------------------
🟢 KẾT QUẢ: BÌNH THƯỜNG (Outcome=0)
📊 Độ tin cậy: 99.86%
💡 Lời khuyên: Chưa thấy dấu hiệu bệnh rõ ràng.
   Hãy duy trì lối sống lành mạnh nhé.

--------------------------------------------------
📝 INPUT: ""Mặc dù tôi ăn rất nhiều đồ ngọt và béo nhưng đi khám sức khỏe định kỳ mọi chỉ số đều tốt.""
--------------------------------------------------
🔴 KẾT QUẢ: CÓ NGUY C